In [ ]:
!pip install -q openai

In [ ]:
import os
from google.colab import userdata
from openai import OpenAI
client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

print("Client initialized successfully!")

In [ ]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "system", "content": "You are a funny joker"},
        {"role": "user", "content": "Tell me a joke"},
    ],
)

print(response.choices[0].message.content)

In [ ]:
SYSTEM = """ROLE: You are the assistant for Sweet Crumbs Home Bakery.
FACTS: We bake to order. Orders need 24 hours' notice. All cakes have eggless versions.
        Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80.
RULES: Never promise a delivery time. If the answer isn't in FACTS, say "I'll check with the owner."
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": "Can I get an eggless cake today evening?"},
    ],
)

print(response.choices[0].message.content)
print(response.usage)

In [ ]:
import json, functools
from openai import RateLimitError, APIConnectionError

MODEL = "openai/gpt-oss-20b"
MENU = {"plum cake": 500, "chocolate cake": 650, "
": 80}

SYSTEM = f"""ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Menu with prices in rupees: {json.dumps(MENU)}
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time. If it isn't in FACTS, say "I'll check with the owner."
        Remember the customer's name and order during the chat.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper

@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for qty of an item on the menu."""
    return MENU[item.lower().strip()] * qty

history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8

def chat(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    recent = [history[0]] + history[1:][-MAX_TURNS:]
    try:
        response = client.chat.completions.create(model=MODEL, messages=recent, temperature=0.3)
        reply = response.choices[0].message.content
    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."
    except APIConnectionError:
        reply = "I can't reach the server right now. Please try again shortly."
    history.append({"role": "assistant", "content": reply})
    return reply

print("Total for 6 cupcakes:", get_price("cupcake", 6))
while True:
    text = input("You: ")
    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting Sweet Crumbs!")
        break
    print("Bot:", chat(text))

In [ ]:
history


In [ ]:
# AI Study Buddy – Project Description
# An AI-powered chatbot designed to help students learn Data Structures and Algorithms (DSA), Python, C++, Java, and Computer Science concepts. It provides simple explanations, coding examples, quizzes, and step-by-step guidance. It supports three learning modes: Explain Mode, Quiz Mode, and Code Mode, with conversation memory for a better learning experience.
# Technology Used: Python, OpenAI SDK, Groq API, GPT-OSS-20B.

In [ ]:

import functools

MODEL = "openai/gpt-oss-20b"

SYSTEM = """ROLE: You are an intelligent AI Study Buddy for Computer Science students.

PURPOSE:
Help students learn DSA, Python, C++, Java, programming and computer science concepts.

RULES:
1. Explain concepts in simple English.
2. Give suitable examples for every concept.
3. Explain code step by step when requested.
4. For coding problems, provide clear and beginner-friendly solutions.
5. Ask small practice questions when the student wants a quiz.
6. Never provide incorrect information intentionally.
7. If you are unsure about an answer, clearly mention it.

FORMAT:
Keep answers organized, friendly and easy to understand.
Adapt explanations to the student's learning level.
"""

# Different learning modes
MODES = {
    "explain": "Explain concepts from basics with examples.",
    "quiz": "Ask one question at a time, wait for the answer, and evaluate it.",
    "code": "Focus on programming examples, code and dry runs."
}

# Logging decorator
def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper


# Get learning mode instructions
@log_call
def get_mode_instruction(mode: str) -> str:
    """Return instructions for the selected learning mode."""
    return MODES.get(
        mode.lower().strip(),
        MODES["explain"]
    )


# Conversation memory
history = [{"role": "system", "content": SYSTEM}]

MAX_MESSAGES = 12


# Chatbot function
def chat(user_text: str, mode: str = "explain") -> str:

    history.append({
        "role": "user",
        "content": user_text
    })

    recent = history[1:][-MAX_MESSAGES:]

    messages = [
        history[0],
        {
            "role": "system",
            "content": "CURRENT LEARNING MODE: " +
                       get_mode_instruction(mode)
        }
    ] + recent

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=0.3
        )

        reply = response.choices[0].message.content or \
                "Sorry, I couldn't generate a response."

    except RateLimitError:
        reply = "The service is busy. Please try again in a minute."

    except APIConnectionError:
        reply = "Unable to connect to the server. Please try again."

    history.append({
        "role": "assistant",
        "content": reply
    })

    return reply


# Start chatbot
print("======================================")
print("       AI STUDY BUDDY")
print("======================================")
print("Available modes: explain, quiz, code")
print("Change mode using: /mode quiz")
print("Type 'bye' to exit.")
print("--------------------------------------")

current_mode = "explain"

while True:

    user_text = input("You: ").strip()

    if user_text.lower() in ("bye", "exit", "quit"):
        print("Study Buddy: Happy learning! Keep practicing.")
        break

    # Change learning mode
    if user_text.lower().startswith("/mode "):

        new_mode = user_text[6:].strip().lower()

        if new_mode in MODES:
            current_mode = new_mode
            print("Study Buddy: Mode changed to", current_mode)
        else:
            print("Available modes:", ", ".join(MODES.keys()))

        continue

    if not user_text:
        continue

    print("Study Buddy:", chat(user_text, current_mode))
